# RSA Key Generation & Encryption: Lockbox Simulator

RSA is a public-key cryptography algorithm. One key can be shared openly for encryption, while the matching private key is kept secret for decryption.

Rivest, Shamir, and Adleman introduced RSA in 1977, turning number theory into a practical tool for digital trust. Public-key cryptography became part of secure web traffic, software updates, digital signatures, certificates, and private communication.

In this notebook, we will build a **toy RSA system** with small numbers so the math is visible.

<details>
<summary>Important security note</summary>

This notebook is for learning only. Real RSA uses huge primes, careful padding schemes, secure randomness, and battle-tested libraries.

</details>

## 1. The Mental Model

RSA turns number theory into a lockbox:

- Public key: anyone can use it to lock a message.
- Private key: only the owner can unlock it.
- Modulus `n`: the shared number both keys use.
- Exponent `e`: the public locking exponent.
- Exponent `d`: the private unlocking exponent.

The core operations are:

```text
encrypt: c = m^e mod n
decrypt: m = c^d mod n
```

<details>
<summary>Why primes matter</summary>

RSA starts with two primes `p` and `q`. Multiplying them is easy. Factoring their product `n = p * q` is hard when the primes are huge.

</details>

## 2. Build the Objects

Implementation plan:

1. `PrimePair` stores the two toy primes.
2. `PublicKey` stores `(e, n)`.
3. `PrivateKey` stores `(d, n)`.
4. `RSAKeyPair` bundles everything together.
5. `KeyStep` records each key-generation step.
6. `RSAWorkshop` owns the math helpers, encryption, and decryption.
7. `RSAReplay` prints the key-generation story.

<details>
<summary>Modular inverse hint</summary>

The private exponent `d` is chosen so `e * d` leaves remainder `1` when divided by `phi`. In Python terms: `(e * d) % phi == 1`.

</details>

**Object model.** Define `PrimePair`, `PublicKey`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

from math import gcd, isqrt

@dataclass(frozen=True)
class PrimePair:
    p: int
    q: int

    def __str__(self) -> str:
        return f"p={self.p}, q={self.q}"

@dataclass(frozen=True)
class PublicKey:
    e: int
    n: int

    def __str__(self) -> str:
        return f"public key (e={self.e}, n={self.n})"


**Trace model.** Define `PrivateKey`, `RSAKeyPair`, `KeyStep`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass(frozen=True)
class PrivateKey:
    d: int
    n: int

    def __str__(self) -> str:
        return f"private key (d={self.d}, n={self.n})"

@dataclass(frozen=True)
class RSAKeyPair:
    primes: PrimePair
    phi: int
    public: PublicKey
    private: PrivateKey

@dataclass
class KeyStep:
    name: str
    value: str
    note: str


**Algorithm engine.** Define `RSAWorkshop`, the class that runs the main simulation or algorithm.


In [ ]:
class RSAWorkshop:
    def __init__(self):
        self.steps: list[KeyStep] = []

    def is_prime(self, number: int) -> bool:
        if number < 2:
            return False
        if number == 2:
            return True
        if number % 2 == 0:
            return False
        for factor in range(3, isqrt(number) + 1, 2):
            if number % factor == 0:
                return False
        return True

    def extended_gcd(self, a: int, b: int) -> tuple[int, int, int]:
        if b == 0:
            return a, 1, 0

        gcd_value, x1, y1 = self.extended_gcd(b, a % b)
        x = y1
        y = x1 - (a // b) * y1
        return gcd_value, x, y

    def modular_inverse(self, number: int, modulus: int) -> int:
        gcd_value, x, _ = self.extended_gcd(number, modulus)
        if gcd_value != 1:
            raise ValueError(f"{number} has no modular inverse mod {modulus}.")
        return x % modulus

    def generate_keys(self, p: int, q: int, e: int) -> RSAKeyPair:
        if p == q:
            raise ValueError("RSA needs two different primes.")
        if not self.is_prime(p) or not self.is_prime(q):
            raise ValueError("Both p and q must be prime.")

        self.steps = []
        primes = PrimePair(p, q)
        n = p * q
        phi = (p - 1) * (q - 1)

        if not 1 < e < phi:
            raise ValueError("The public exponent e must be between 1 and phi.")
        if gcd(e, phi) != 1:
            raise ValueError("The public exponent e must be coprime with phi.")

        d = self.modular_inverse(e, phi)
        key_pair = RSAKeyPair(
            primes=primes,
            phi=phi,
            public=PublicKey(e=e, n=n),
            private=PrivateKey(d=d, n=n),
        )

        self.steps.extend(
            [
                KeyStep("Choose primes", str(primes), "Start with two toy primes."),
                KeyStep("Compute n", f"n = {p} * {q} = {n}", "The modulus is shared by both keys."),
                KeyStep("Compute phi", f"phi = ({p} - 1) * ({q} - 1) = {phi}", "This counts values that behave well with n."),
                KeyStep("Choose e", f"e = {e}", "e must be coprime with phi."),
                KeyStep("Solve for d", f"d = {d}", "d is the modular inverse of e mod phi."),
                KeyStep("Build keys", f"{key_pair.public}; {key_pair.private}", "Public encrypts, private decrypts."),
            ]
        )
        return key_pair

    def encrypt_number(self, message_number: int, public_key: PublicKey) -> int:
        if not 0 <= message_number < public_key.n:
            raise ValueError("Message number must be between 0 and n - 1.")
        return pow(message_number, public_key.e, public_key.n)

    def decrypt_number(self, cipher_number: int, private_key: PrivateKey) -> int:
        return pow(cipher_number, private_key.d, private_key.n)

    def encrypt_text(self, text: str, public_key: PublicKey) -> list[int]:
        return [self.encrypt_number(ord(character), public_key) for character in text]

    def decrypt_text(self, cipher_blocks: list[int], private_key: PrivateKey) -> str:
        return "".join(chr(self.decrypt_number(block, private_key)) for block in cipher_blocks)


## 3. Generate a Toy Key Pair

We will use small classic demo primes so the numbers stay readable:

- `p = 61`
- `q = 53`
- `e = 17`

<details>
<summary>Why these numbers work</summary>

`p` and `q` are prime, and `e = 17` is coprime with `phi = (p - 1)(q - 1)`. That means `e` has a modular inverse, which becomes `d`.

</details>

In [2]:
workshop = RSAWorkshop()
key_pair = workshop.generate_keys(p=61, q=53, e=17)

print(key_pair.public)
print(key_pair.private)
print(f"phi = {key_pair.phi}")
print(f"Check inverse: (e * d) % phi = {(key_pair.public.e * key_pair.private.d) % key_pair.phi}")

public key (e=17, n=3233)
private key (d=2753, n=3233)
phi = 3120
Check inverse: (e * d) % phi = 1


## 4. Encrypt and Decrypt a Number

RSA works on numbers. Text must be converted into numbers first, but the core operation is numeric.

<details>
<summary>Math check</summary>

The message number must be smaller than `n`. In this toy key pair, `n = 3233`, so a message like `42` is valid.

</details>

In [3]:
message_number = 42
cipher_number = workshop.encrypt_number(message_number, key_pair.public)
decrypted_number = workshop.decrypt_number(cipher_number, key_pair.private)

print(f"Message number:   {message_number}")
print(f"Encrypted number: {cipher_number}")
print(f"Decrypted number: {decrypted_number}")
print(f"Round trip worked: {message_number == decrypted_number}")

Message number:   42
Encrypted number: 2557
Decrypted number: 42
Round trip worked: True


## 5. Encrypt a Short Message

For this toy notebook, we encrypt each character code separately. That keeps each block smaller than `n` and makes the transformation easy to inspect.

<details>
<summary>Real-world difference</summary>

Real RSA does not encrypt plain characters directly. It uses padding and carefully designed encoding rules before modular exponentiation.

</details>

In [4]:
secret_message = "MATH"
cipher_blocks = workshop.encrypt_text(secret_message, key_pair.public)
opened_message = workshop.decrypt_text(cipher_blocks, key_pair.private)

print(f"Original message:  {secret_message}")
print(f"Character codes:   {[ord(character) for character in secret_message]}")
print(f"Cipher blocks:     {cipher_blocks}")
print(f"Decrypted message: {opened_message}")

Original message:  MATH
Character codes:   [77, 65, 84, 72]
Cipher blocks:     [3123, 2790, 2159, 3000]
Decrypted message: MATH


## 6. Replay Key Generation

RSA key generation has several moving pieces. The replay shows each number being created in order.

<details>
<summary>Reading the replay</summary>

The most important check is the inverse: `d` must undo `e` modulo `phi`.

</details>

In [5]:
class RSAReplay:
    def __init__(self, steps: list[KeyStep]):
        self.steps = steps

    def show(self) -> None:
        for step_number, step in enumerate(self.steps, start=1):
            print(f"Step {step_number}: {step.name}")
            print(f"  value: {step.value}")
            print(f"  note:  {step.note}\n")


RSAReplay(workshop.steps).show()

Step 1: Choose primes
  value: p=61, q=53
  note:  Start with two toy primes.

Step 2: Compute n
  value: n = 61 * 53 = 3233
  note:  The modulus is shared by both keys.

Step 3: Compute phi
  value: phi = (61 - 1) * (53 - 1) = 3120
  note:  This counts values that behave well with n.

Step 4: Choose e
  value: e = 17
  note:  e must be coprime with phi.

Step 5: Solve for d
  value: d = 2753
  note:  d is the modular inverse of e mod phi.

Step 6: Build keys
  value: public key (e=17, n=3233); private key (d=2753, n=3233)
  note:  Public encrypts, private decrypts.



## 7. Experiments

Small changes can break RSA or change the ciphertext completely.

<details>
<summary>Experiment hint</summary>

RSA depends on exact modular arithmetic. A bad exponent, a changed ciphertext, or weak primes changes the story fast.

</details>

In [6]:
try:
    RSAWorkshop().generate_keys(p=61, q=53, e=12)
except ValueError as error:
    print(f"Bad exponent experiment: {error}")


tampered_cipher = (cipher_number + 1) % key_pair.public.n
tampered_plain = workshop.decrypt_number(tampered_cipher, key_pair.private)

print("\nTampered ciphertext experiment:")
print(f"original cipher:  {cipher_number}")
print(f"tampered cipher:  {tampered_cipher}")
print(f"decrypted result: {tampered_plain}")
print(f"still the message: {tampered_plain == message_number}")

Bad exponent experiment: The public exponent e must be coprime with phi.

Tampered ciphertext experiment:
original cipher:  2557
tampered cipher:  2558
decrypted result: 22
still the message: False


In [7]:
alternate_workshop = RSAWorkshop()
alternate_keys = alternate_workshop.generate_keys(p=47, q=59, e=13)
alternate_message = "DATA"
alternate_cipher = alternate_workshop.encrypt_text(alternate_message, alternate_keys.public)
alternate_opened = alternate_workshop.decrypt_text(alternate_cipher, alternate_keys.private)

print("Alternate key pair:")
print(alternate_keys.public)
print(alternate_keys.private)
print(f"cipher blocks: {alternate_cipher}")
print(f"decrypted: {alternate_opened}")

Alternate key pair:
public key (e=13, n=2773)
private key (d=821, n=2773)
cipher blocks: [1746, 660, 902, 660]
decrypted: DATA


In [8]:
def factor_toy_modulus(n: int) -> PrimePair | None:
    for possible_factor in range(2, isqrt(n) + 1):
        if n % possible_factor == 0:
            return PrimePair(possible_factor, n // possible_factor)
    return None


recovered_primes = factor_toy_modulus(key_pair.public.n)

print("Toy factoring experiment:")
print(f"public n: {key_pair.public.n}")
print(f"recovered primes: {recovered_primes}")
print("Tiny primes are easy to recover. Real RSA uses primes with hundreds of digits.")

Toy factoring experiment:
public n: 3233
recovered primes: p=53, q=61
Tiny primes are easy to recover. Real RSA uses primes with hundreds of digits.


## What You Should Remember

RSA key generation builds two linked keys from prime-number math:

- `n = p * q` becomes the shared modulus.
- `phi = (p - 1)(q - 1)` helps create the private exponent.
- `e` is public and must be coprime with `phi`.
- `d` is private and satisfies `(e * d) % phi == 1`.
- Encryption is `m^e mod n`; decryption is `c^d mod n`.

<details>
<summary>Where this shows up</summary>

RSA appears in secure communication, certificates, signatures, and key exchange history. Modern systems often combine public-key cryptography with faster symmetric encryption.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Build public-key cryptography from modular arithmetic and factoring hardness.

**Interactive animation target.** Animate modular exponentiation as repeated squaring and show key generation dependencies.

**Correctness handle.** The public/private exponents are modular inverses modulo phi(n) or lambda(n).

**Complexity handle.** Modular exponentiation is polynomial in bit length; factoring large n is the hard classical step.

**Failure mode to test.** Small primes, bad padding, or reused structure can break RSA even without factoring.

**Studio task.** Generate a tiny key, encrypt one message, and trace each repeated-squaring step.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
